## How To Interpret:
- `interactions.csv`: Contains the anonymized follower edge list. Each row consists of two comma-separated integers representing a directed following relation (i.e., user u follows user v).
- `followers.csv`: Contains the anonymized interactions edge list. Each row consists of six comma-separated integers representing a comment, repost, or quote interaction. These integers correspond to the following fields, in this order: user_id, replied_author, thread_root_author, reposted_author,quoted_author, and date.

In [2]:
import igraph as ig
import leidenalg as la
import polars as pl

from bluesky_case_study import load

In [2]:
followers_df = load.load_followers().collect()

followers_df.shape
followers_df.head()
followers_df.sample(10)  # random rows are more honest than the first rows

src,dst
i64,i64
111494,10702
153027,14874
83250,275107
224747,28113
429392,703142
3551028,421
3866805,168
328978,1046339
157908,194962


In [2]:
interactions_df = load.load_interactions().collect()

interactions_df.shape
interactions_df.head()
interactions_df.sample(10)

src,replied,root,reposted,quoted,date
i64,i64,i64,i64,i64,datetime[μs]
407308,617611,617611,null,null,2023-11-05 20:59:00
805450,805450,805450,null,805450,2023-10-03 05:54:00
33002,67133,33002,null,null,2024-01-10 13:37:00
180750,null,null,71768,null,2023-12-08 14:19:00
106636,14066,14066,null,null,2024-03-02 06:44:00
675192,null,null,38593,null,2024-03-05 14:30:00
341561,72,72,30319,null,2023-10-09 13:31:00
566583,1191,1191,null,null,2023-09-29 12:35:00
96960,null,null,217042,null,2024-03-04 20:02:00


In [ ]:
weighted_g = load.load_weighted().collect()

src,dst,replies,reposts,quotes,weight
i64,i64,u32,u32,u32,f64
2083949,143404,0,0,0,0.0
2083949,158654,0,0,0,0.0
2083975,19399,0,0,0,0.0
2083975,35190,0,1,0,0.693147
2083979,2079705,0,0,0,0.0


In [ ]:
G = load.load_weighted_graph()
partition = la.find_partition(G, la.ModularityVertexPartition);

## Community detection on a sample (top 100k users)

The top 100k users by strength are the network's most active core, so quieter communities may not show up here. Check conclusions on the full active graph.

In [ ]:
%load_ext autoreload
%autoreload 2

import random

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from bluesky_case_study import communities as cm

In [ ]:
# Sample graph: the 100k users with the highest strength (weight sent + received)
g_sample, users = load.load_top_k_graph(100_000)
g_sample.summary()

In [ ]:
# Leiden with igraph's built-in version (about 10s here; leidenalg took several minutes on this sample).
# Treats the graph as undirected: u→v and v→u become one edge with their weights added.
g_undirected = g_sample.as_undirected(mode="collapse", combine_edges="sum")


def run_leiden(seed: int) -> list[int]:
    random.seed(seed)  # igraph uses Python's random module
    part = g_undirected.community_leiden(
        objective_function="modularity", weights="weight", n_iterations=-1
    )
    return cm.relabel_by_size(part.membership)  # C0 = largest community


membership = run_leiden(42)
print(
    f"{max(membership) + 1} communities, modularity "
    f"{g_undirected.modularity(membership, weights='weight'):.3f}"
)

In [ ]:
# One row per community: size, share of users, internal_share (weight that stays
# inside the community), and its 5 highest-strength users
summary = cm.community_summary(g_sample, membership, users)
with pl.Config(tbl_rows=15, fmt_str_lengths=60):
    display(summary.head(15))

In [ ]:
# Community sizes: 15 largest, the rest folded into "Other"
TOP = 15
sizes = summary["size"].to_list()
labels = [f"C{c}" for c in summary["community"][:TOP]] + [f"Other ({len(sizes) - TOP})"]
vals = sizes[:TOP] + [sum(sizes[TOP:])]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(labels[::-1], vals[::-1], color="#2a78d6", height=0.7)
for y, v in enumerate(vals[::-1]):
    ax.text(v, y, f" {v:,}", va="center", fontsize=8, color="#52514e")
ax.set_title("Community sizes (top 100k users)", loc="left")
ax.set_xlabel("Users")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#e5e4df", lw=0.6)
ax.set_axisbelow(True)
plt.show()

In [ ]:
# Where each community's interaction weight goes (10 largest communities)
K = 10
keep = summary["community"][:K].to_list()
R = cm.mixing_matrix(g_sample, membership, keep)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(R, cmap="Blues", vmin=0, vmax=1)
names = [f"C{c}" for c in keep]
ax.set_xticks(range(K), names)
ax.set_yticks(range(K), names)
for i in range(K):
    for j in range(K):
        if R[i, j] >= 0.01:
            ax.text(
                j,
                i,
                f"{R[i, j]:.0%}",
                ha="center",
                va="center",
                fontsize=7,
                color="white" if R[i, j] > 0.5 else "#0b0b0b",
            )
ax.set_title("Share of each community's interaction weight", loc="left")
ax.set_ylabel("From community")
ax.set_xlabel("To community")
fig.colorbar(im, ax=ax, fraction=0.04, format=PercentFormatter(1.0))
plt.show()

In [ ]:
# Stability: rerun with other seeds and compare. NMI of 1.0 means identical communities
other_runs = [run_leiden(seed) for seed in (1, 2)]
[
    round(ig.compare_communities(membership, other, method="nmi"), 3)
    for other in other_runs
]